# Does vulnerable employment increase during a recession?

**ECON 630 project.** This notebook follows the plan in [`Claude.MD`](Claude.MD). All data are pulled from the World Bank API inside the notebook, so anyone can re-run it top to bottom.

**Hypothesis:** Vulnerable employment increases during a recession because formal employment opportunities dry up, pushing workers into own-account work and contributing family work.

**Plan of the notebook**

1. Setup
2. Pull the data from the World Bank API (`wbgapi`)
3. Cross-check the API pull against the CSVs saved in the ECON630 folder
4. Clean: drop regional/income aggregates, reshape to a country–year panel
5. Define recessions and the outcome variable
6. Descriptive comparison: recession vs. non-recession years
7. Panel regression with country and year fixed effects
8. Event study around recession onsets
9. Heterogeneity by income group
10. Conclusions and caveats

## 1. Setup

In [1]:
from pathlib import Path
import re
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
import wbgapi as wb

START, END = 1991, 2025          # ILO modeled estimates start in 1991

# Optional saved World Bank CSVs (repo root or this folder), used only for the cross-check in step 3
DATA_DIR = next((p for p in [Path.cwd(), Path.cwd().parent]
                 if any(p.glob("API_SL.EMP.VULN.ZS*.zip"))), None)

FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
SOURCE = "Source: World Bank WDI via wbgapi (SL.EMP.VULN.ZS, modeled ILO estimate; NY.GDP.MKTP.KD.ZG)."

BLUE, ORANGE, MUTED = "#2a78d6", "#eb6834", "#898781"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": "#333", "xtick.color": MUTED,
    "ytick.color": MUTED, "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "lines.linewidth": 2, "axes.axisbelow": True,
})
pd.set_option("display.float_format", "{:.3f}".format)

def finish(fig, name):
    # add the data-source caption, save to figures/, and display

    fig.text(0.01, -0.03, SOURCE, fontsize=8, color=MUTED, ha="left", va="top")
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=150)
    plt.show()

### API keys

The World Bank API is free and doesn't need a key, so this project has no secrets. If a key-protected source is added later, put the key in a `.env` file (already excluded by `.gitignore`) and read it with `os.getenv`. Keys never go in the notebook or the repo.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()   # loads variables from a local .env file if one exists; does nothing otherwise
# e.g. FRED_KEY = os.getenv("FRED_API_KEY")   # not needed for the World Bank API

False

## 2. Pull the data from the World Bank API

| Code | Indicator | Role |
|---|---|---|
| `SL.EMP.VULN.ZS` | Vulnerable employment, % of total employment (modeled ILO estimate) | **Outcome** |
| `NY.GDP.MKTP.KD.ZG` | GDP growth, annual %, constant prices | **Recession indicator** |
| `NY.GDP.MKTP.CD` | GDP, current US$ | Matches the saved CSV; used only for the cross-check |

The recession indicator is **real** GDP growth. Current-US$ GDP moves with exchange rates and inflation, so a fall in nominal dollar GDP doesn't mean the economy contracted.

In [3]:
SERIES = {
    "SL.EMP.VULN.ZS":    "vuln",
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",
    "NY.GDP.MKTP.CD":    "gdp_usd",
}

economies = wb.economy.DataFrame()
raw = wb.data.DataFrame(list(SERIES), economy="all", time=range(START, END + 1),
                        columns="series").reset_index()
raw.head()

,economy,time,NY.GDP.MKTP.CD,NY.GDP.MKTP.KD.ZG,SL.EMP.VULN.ZS
0,ABW,YR1991,872067039.106,7.963,NaN
1,ABW,YR1992,958659217.877,5.882,NaN
2,ABW,YR1993,1083240223.464,7.308,NaN
3,ABW,YR1994,1245810055.866,8.204,NaN
4,ABW,YR1995,1320670391.061,2.547,NaN


## 3. Cross-check against the CSVs in the ECON630 folder

The zipped World Bank CSVs are wide (one column per year) and have four lines of header metadata. If they agree with the API pull, we know we're using the same data. This step is **optional**: the analysis uses only the API data, and the step is skipped if the zip files aren't present.

In [4]:
def load_wb_zip(path, name):
    with zipfile.ZipFile(path) as z:
        csv = next(f for f in z.namelist() if f.startswith("API_"))
        wide = pd.read_csv(z.open(csv), skiprows=4)
    year_cols = [c for c in wide.columns if c.isdigit()]
    return (wide.melt(id_vars="Country Code", value_vars=year_cols, var_name="year", value_name=name)
                .rename(columns={"Country Code": "iso3"})
                .astype({"year": int}))

if DATA_DIR is None:
    print("Saved CSV zips not found; skipping the cross-check (the analysis uses only the API pull).")
else:
    local = load_wb_zip(next(DATA_DIR.glob("API_SL.EMP.VULN.ZS*.zip")), "vuln_local").merge(
            load_wb_zip(next(DATA_DIR.glob("API_NY.GDP.MKTP.CD*.zip")), "gdp_usd_local"),
            on=["iso3", "year"], how="outer")

    api_check = raw.assign(year=raw["time"].str[2:].astype(int)).rename(columns={"economy": "iso3"})
    chk = api_check.merge(local, on=["iso3", "year"], how="inner")

    for api_col, loc_col in [("SL.EMP.VULN.ZS", "vuln_local"), ("NY.GDP.MKTP.CD", "gdp_usd_local")]:
        both = chk[[api_col, loc_col]].dropna()
        rel = ((both[api_col] - both[loc_col]).abs() / both[loc_col].abs()).max()
        print(f"{api_col:16s} overlapping obs = {len(both):5d}   max relative difference = {rel:.2e}")

SL.EMP.VULN.ZS   overlapping obs =  8176   max relative difference = 0.00e+00
NY.GDP.MKTP.CD   overlapping obs =  8854   max relative difference = 0.00e+00


## 4. Clean the data

* Convert `YR2009` → `2009`.
* **Drop aggregates.** The World Bank bundles regions and income groups ("World", "Euro area", "Low income", …) in with countries. Leaving them in would double-count countries.
* Attach country name and income group.
* Keep countries with at least 10 years of both variables.

In [5]:
countries = economies.loc[~economies["aggregate"], ["name", "incomeLevel", "region"]]

panel = (raw.rename(columns=SERIES | {"economy": "iso3"})
            .assign(year=lambda d: d["time"].str[2:].astype(int))
            .drop(columns="time")
            .merge(countries, left_on="iso3", right_index=True, how="inner")
            .sort_values(["iso3", "year"])
            .reset_index(drop=True))

print(f"Rows before filtering: {len(panel):,}   countries: {panel['iso3'].nunique()}")

n_obs = panel.dropna(subset=["vuln", "gdp_growth"]).groupby("iso3").size()
panel = panel[panel["iso3"].isin(n_obs[n_obs >= 10].index)].copy()

print(f"Rows after filtering:  {len(panel):,}   countries: {panel['iso3'].nunique()}")
print(f"Years covered: {panel.dropna(subset=['vuln'])['year'].min()}–{panel.dropna(subset=['vuln'])['year'].max()}")
panel[["vuln", "gdp_growth"]].describe().T

Rows before filtering: 7,595   countries: 217
Rows after filtering:  6,475   countries: 185
Years covered: 1991–2025


,count,mean,std,min,25%,50%,75%,max
vuln,6463.000,37.459,26.497,0.200,13.743,32.275,58.862,95.530
gdp_growth,6325.000,3.408,6.765,-64.047,1.343,3.642,5.909,149.973


## 5. Define recessions and the outcome

* **Recession year:** real GDP growth < 0. With annual data this is the standard cross-country definition; the NBER-style quarterly definition isn't available for most countries.
* **Recession onset:** a recession year that follows a non-recession year.
* **Outcome:** the year-over-year change in vulnerable employment, in percentage points: $\Delta vuln_{it} = vuln_{it} - vuln_{i,t-1}$. The hypothesis is about vulnerable employment *increasing*, and differencing also strips out each country's permanent level (for example, Niger ≈ 90% vs. Germany ≈ 6%).

In [6]:
g = panel.groupby("iso3")
panel["recession"]     = (panel["gdp_growth"] < 0).astype(float).where(panel["gdp_growth"].notna())
panel["recession_lag"] = g["recession"].shift(1)
panel["onset"]         = ((panel["recession"] == 1) & (panel["recession_lag"] == 0)).astype(int)
panel["d_vuln"]        = g["vuln"].diff()
panel["d_vuln_lead"]   = g["d_vuln"].shift(-1)   # change in the year after

reg = panel.dropna(subset=["d_vuln", "recession"])
print(f"Country-years with outcome + recession flag: {len(reg):,}")
print(f"Recession country-years: {int(reg['recession'].sum()):,} ({reg['recession'].mean():.1%})")
print(f"Recession onsets:        {int(reg['onset'].sum()):,}")

Country-years with outcome + recession flag: 6,138
Recession country-years: 950 (15.5%)
Recession onsets:        573
